# MBC 뉴스 URL 수집 — 카테고리×일자 / 기간 모드 (Colab용)

MBC 뉴스 사이트(`imnews.imbc.com`)는 카테고리별 일자 페이지 구조다. 일자 페이지 HTML(`{Link}{CurrentID}_{TempleteId}.html`)은 첫 10건 정도만 노출하고 나머지는 "더보기" 버튼이 호출하는 데이터 JSON(`{Link}{CurrentID}_{DataId}.js`)에 들어있다. 우리는 그 데이터 JSON을 직접 받아 **카테고리×일자별 전체 기사 리스트**를 한 번에 확보한다 (페이지 HTML 받기보다 약 4배 많은 기사).

URL 빌드는 카테고리×연도별 `cal_data.js` (`/news/{YYYY}/{category}/cal_data.js`)가 제공하는 일자→CurrentID/TempleteId/DataId 매핑을 사용한다.

통신3사/LPOD/SBS/KBS 트랙과 동일하게 **기간 단위로 통합 JSON** 1개를 만들고, 내부에서 일자별로 6개 카테고리(정치/사회/국제/경제/문화/스포츠)를 순회한다.

- 입력: `press_ranges` — 언론사(press) + 기간(start_date, end_date)
- 출력: `data/링크_{press}_{YYMMDD}_{YYMMDD}.json` (기간 통합 기사 URL)
- 보조 출력: 기간별 수집 로그 JSON, 중간 재개용 temp JSON
- 특징: 카테고리×연도별 cal_data.js 1회 캐싱, 일자별 6 카테고리 JSON 데이터 직접 호출, BS4 불필요 (JSON 파싱만)
- 트랙 B(네이버 경유, press='MBC')와 파일명 충돌 방지를 위해 press 이름에 `_direct` 접미사 사용
- URL 빌드 출처: bundle.min.js의 `ImCommon.GetArtListUrl(catId, ..., "J", cb)` → `{Link}{CurrentID}_{DataId}.js`


In [1]:
# Colab 환경 세팅 — requests, BeautifulSoup 설치 (Selenium 불필요, MBC는 정적 HTML)
# !pip install -q requests beautifulsoup4

In [2]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Drive 안의 프로젝트 폴더로 이동
# 통신3사 트리와 분리하기 위해 KBS/SBS/MBC 등 언론사 직접 수집은 news/ 하위에 보관
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')

현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news


In [6]:
import json
import os
import time
from datetime import datetime, timedelta
from pathlib import Path

import requests
from bs4 import BeautifulSoup

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 언론사와 수집 기간 지정 — 날짜 형식: 'YYYY.MM.DD'
# press 하나당 start_date ~ end_date 안의 일자를 한 통합 JSON으로 묶어 저장 (통신3사/LPOD/SBS/KBS 트랙과 동일 패턴)
# 트랙 B(네이버 경유, press='MBC') 결과와 파일명 충돌 방지를 위해 press 이름에 _direct 접미사 사용
press_ranges = [
    {'press': 'MBC_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]

# MBC 카테고리 매핑 — bundle.min.js의 szCategory 테이블에서 발췌
# 키는 URL 카테고리 segment, 값은 한글 라벨 (수집 로그/디버깅용)
# 6개 모두 'D'(일자별) 타입이라 cal_data.js로 일자 매핑 받음
CATEGORIES = {
    'politics': '정치',
    'society': '사회',
    'world': '국제',
    'econo': '경제',
    'culture': '문화',
    'sports': '스포츠',
}


# 기간 단위 작업 목록 생성 — press_ranges 한 항목당 jobs 1개
# 카테고리/일자 순회는 collect_links_for_period 내부에서 처리
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 문자열을 datetime으로 파싱 — 기간 유효성 검사용
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        jobs.append({
            'press': item['press'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 생성된 jobs는 다음 셀에서 순서대로 실행
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)
print(f'카테고리: {list(CATEGORIES.keys())} ({len(CATEGORIES)}개)')

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'

# 저장할 폴더 지정 — 링크 파일, 수집 로그, 임시 체크포인트, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = Path(PROJECT_DIR) / 'notebook' / 'crawling' / 'data'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

# requests 세션 생성 — User-Agent / 언어 / Referer 헤더를 반복 요청에 일관 적용
session = requests.Session()
session.headers.update({
    'User-Agent': USER_AGENT,
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://imnews.imbc.com/news/',
})
print(f'User-Agent: {USER_AGENT}')

총 작업 수: 1
{'press': 'MBC_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
카테고리: ['politics', 'society', 'world', 'econo', 'culture', 'sports'] (6개)
저장 위치: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36


In [ ]:
import random
import re

# 서버 부담을 줄이기 위해 페이지/일자/job 사이에 랜덤 대기
PAGE_PAUSE_RANGE_SEC = (0.4, 1.2)
DAY_PAUSE_RANGE_SEC = (2, 5)
JOB_PAUSE_RANGE_SEC = (5, 12)
REQUEST_TIMEOUT_SEC = 15
SKIP_COMPLETED = True

MBC_BASE = 'https://imnews.imbc.com'
# 기사 URL 패턴 — JSON Link 필드는 카테고리 외에도 /replay/{YYYY}/{program}/article/... 같이 다양한 경로 포함
# 본문 노트북이 처리할 수 있는 imnews.imbc.com 안 article 페이지 전부 허용
ARTICLE_URL_PATTERN = re.compile(r'^https://imnews\.imbc\.com/[\w/]+/article/\d+_\d+\.html$')


def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


def build_cal_url(year, category):
    return f'{MBC_BASE}/news/{year}/{category}/cal_data.js'


# cal_data.js 받아 JSON 파싱 — BOM(﻿) 제거 후 json.loads
def fetch_cal_data(year, category):
    response = session.get(build_cal_url(year, category), timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()
    return json.loads(response.text.lstrip('﻿'))


# 일자 데이터 JSON URL 빌드 — bundle.min.js의 GetArtListUrl(sType="J") 로직 그대로 재현
# 형식: {Link}{CurrentID}_{DataId}.js (예: .../politics/6820020_36910.js)
# 페이지 HTML(_{TempleteId}.html)은 첫 10건 정도만 노출하지만, 데이터 JS(_{DataId}.js)는
# 그 일자 카테고리의 전체 기사 리스트(보통 수십 건)를 한 번에 반환 — 카테고리 페이지의 "더보기" 버튼이
# 호출하는 것과 동일한 엔드포인트
def build_date_data_url(cal, date_ymd):
    for entry in cal['DateList']:
        if entry['Day'] == date_ymd:
            return f"{cal['Link']}{entry['CurrentID']}_{cal['DataId']}.js"
    return None


# 일자 데이터 JSON 한 건 다운로드 (BOM 제거 후 파싱)
# Content-Type이 text/html인 응답(에러 페이지)을 사전 차단해서 JSONDecodeError 대신 명시적 예외 발생
# 일부 카테고리×일자 조합은 cal_data에 entry 있어도 실제 데이터 파일이 비어 있어 에러 HTML로 폴백됨
def fetch_date_data(url):
    response = session.get(url, timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()
    ctype = response.headers.get('Content-Type', '')
    # MBC 데이터 .js는 보통 text/javascript 또는 application/javascript로 옴
    # text/html이면 에러 페이지가 떨어진 것 — 빈 데이터로 처리
    if 'html' in ctype.lower():
        return {'Data': []}
    return json.loads(response.text.lstrip('﻿'))


def extract_article_links_from_data(data):
    links = set()
    for section in data.get('Data', []):
        for article in section.get('List', []):
            link = (article.get('Link') or '').strip()
            if link and ARTICLE_URL_PATTERN.match(link):
                links.add(link)
    return links


# 한 일자의 모든 카테고리 데이터를 순회하며 기사 URL set 반환 + 카테고리별 통계
# 카테고리 단위로 try/except — 한 카테고리에서 오류 나도 다른 카테고리와 이후 일자 계속 진행
def collect_links_for_day(date_ymd, cal_cache):
    day_links = set()
    category_stats = []
    year = date_ymd[:4]

    for cat_key in CATEGORIES:
        key = (year, cat_key)
        # cal_data 캐시 hit/miss — 같은 연도/카테고리는 1회만 다운로드
        # cal_data 자체가 실패하면 그 카테고리 전체 건너뛰고 다음 카테고리로
        if key not in cal_cache:
            try:
                cal_cache[key] = fetch_cal_data(year, cat_key)
                polite_sleep(f"  cal_data {cat_key} 받은 후", PAGE_PAUSE_RANGE_SEC)
            except Exception as exc:
                print(f'  cal_data {cat_key} 받기 실패 (건너뜀): {exc!r}')
                category_stats.append({
                    'category': cat_key, 'data_url': None, 'found': 0, 'added': 0,
                    'error': f'cal_data: {exc!r}',
                })
                continue
        cal = cal_cache[key]

        # 해당 일자의 데이터 URL 빌드 — 일자에 entry 없으면 None
        data_url = build_date_data_url(cal, date_ymd)
        if not data_url:
            category_stats.append({
                'category': cat_key, 'data_url': None, 'found': 0, 'added': 0,
                'note': 'no_date_entry',
            })
            continue

        # JSON 받고 Link 추출 — 카테고리 단위 try/except로 견고화
        # 실제 데이터 파일이 없거나 (에러 HTML) JSON 파싱 실패 시 그 카테고리만 건너뜀
        try:
            data = fetch_date_data(data_url)
            page_links = extract_article_links_from_data(data)
        except Exception as exc:
            print(f'  {cat_key} 데이터 받기 실패 (건너뜀): {exc!r}')
            category_stats.append({
                'category': cat_key, 'data_url': data_url, 'found': 0, 'added': 0,
                'error': repr(exc),
            })
            polite_sleep(f"  {cat_key} 실패 후 대기", PAGE_PAUSE_RANGE_SEC)
            continue

        before = len(day_links)
        day_links.update(page_links)
        category_stats.append({
            'category': cat_key,
            'data_url': data_url,
            'found': len(page_links),
            'added': len(day_links) - before,
            'total_this_day': len(day_links),
        })
        polite_sleep(f"  {cat_key} 다음 카테고리 전", PAGE_PAUSE_RANGE_SEC)

    return day_links, category_stats


# 한 언론사의 기간 전체를 통합 JSON 1개로 저장 (통신3사 collect_links와 동일 패턴)
# 일자별 임시 체크포인트로 중단/재개 지원
def collect_links_for_period(press, start_date, end_date, save_dir=SAVE_DIR):
    start_yymmdd = start_date.replace('.', '')[2:]
    end_yymmdd = end_date.replace('.', '')[2:]
    period = f"{start_yymmdd}_{end_yymmdd}"

    temp_links_path = save_dir / f"{press}_{start_date}_{end_date}_temp_links.json"
    links_save_path = save_dir / f"링크_{press}_{period}.json"
    stats_save_path = save_dir / f"수집로그_{press}_{period}.json"

    if SKIP_COMPLETED and links_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {links_save_path}")
        return links_save_path

    print()
    print(f"=== {press} / {start_date} ~ {end_date} 수집 시작 ===")

    # 임시 파일에 기존 링크가 있으면 불러오기 — last_date 다음 날부터 이어서 수집
    if temp_links_path.exists():
        with temp_links_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        all_links_set = set(checkpoint.get('links', []))
        last_collected_date = checkpoint.get('last_date')
        daily_stats = checkpoint.get('daily_stats', [])
        print(f"기존 임시 파일에서 링크 {len(all_links_set)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집")
    else:
        all_links_set = set()
        last_collected_date = None
        daily_stats = []
        print('새로 링크 수집 시작')

    cal_cache = {}

    current = datetime.strptime(start_date, '%Y.%m.%d')
    end = datetime.strptime(end_date, '%Y.%m.%d')

    while current <= end:
        day_str = current.strftime('%Y.%m.%d')

        if last_collected_date and day_str <= last_collected_date:
            print(f"{day_str} — 이미 수집 완료, 건너뜀")
            current += timedelta(days=1)
            continue

        date_ymd = day_str.replace('.', '')
        started_at = time.time()

        # 하루치 모든 카테고리 순회 — 내부에서 카테고리 단위 try/except로 견고화
        day_links, category_stats = collect_links_for_day(date_ymd, cal_cache)
        before = len(all_links_set)
        all_links_set.update(day_links)
        added = len(all_links_set) - before
        elapsed = round(time.time() - started_at, 2)

        daily_stats.append({
            'date': day_str,
            'unique_in_day': len(day_links),
            'added': added,
            'total': len(all_links_set),
            'elapsed_sec': elapsed,
            'categories': category_stats,
        })

        print(
            f"{day_str} — 일자 내 unique {len(day_links)}건 / 신규 {added}건 "
            f"/ 누적 {len(all_links_set)}건 / {elapsed}초"
        )

        # 하루치 수집 후 임시 파일에 즉시 저장 (중간에 끊겨도 누적 보존 + 마지막 완료 날짜 기록)
        with temp_links_path.open('w', encoding='utf-8') as f:
            json.dump(
                {'links': sorted(all_links_set), 'last_date': day_str, 'daily_stats': daily_stats},
                f, ensure_ascii=False, indent=2,
            )
        last_collected_date = day_str
        current += timedelta(days=1)
        if current <= end:
            polite_sleep('다음 날짜 전', DAY_PAUSE_RANGE_SEC)

    mbc_news_links = sorted(all_links_set)
    with open(links_save_path, 'w', encoding='utf-8') as f:
        json.dump(mbc_news_links, f, ensure_ascii=False, indent=2)

    with open(stats_save_path, 'w', encoding='utf-8') as f:
        json.dump({
            'press': press,
            'start_date': start_date,
            'end_date': end_date,
            'total': len(mbc_news_links),
            'categories_used': list(CATEGORIES.keys()),
            'days': daily_stats,
        }, f, ensure_ascii=False, indent=2)

    if temp_links_path.exists():
        temp_links_path.unlink()

    print(f"수집 완료 — 총 {len(mbc_news_links)}개")
    print(f"링크 저장: {links_save_path}")
    print(f"수집 로그 저장: {stats_save_path}")
    return links_save_path


results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        results.append(collect_links_for_period(**job))
    except Exception as exc:
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어갑니다: {exc!r}")
    finally:
        if index < len(jobs):
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

if failures:
    failures_path = SAVE_DIR / '수집실패목록_MBC_direct.json'
    with open(failures_path, 'w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)
